# FraudFinder - Ingesting Data with Dataflow Streaming

## Overview

This series of labs are updated upon [FraudFinder](https://github.com/googlecloudplatform/fraudfinder) repository which builds a end-to-end real-time fraud detection system on Google Cloud. Throughout the FraudFinder labs, you will learn how to read historical bank transaction data stored in data warehouse, read from a live stream of new transactions, perform exploratory data analysis (EDA), do feature engineering, ingest features into a feature store, train a model using feature store, register your model in a model registry, evaluate your model, deploy your model to an endpoint, do real-time inference on your model with feature store, and monitor your model.


### Objective

As you engineer features for model training, it's important to consider how the features are computed when making predictions with new data. For online predictions, you may have features that can be pre-computed via _batch feature engineering_. You may also features that need to be computed on-the-fly via _streaming-based feature engineering_. For these Fraudfinder labs, for computing features based on the last n _days_, you will use _batch_ feature engineering in BigQuery; for computing features based on the last n _minutes_, you will use _streaming-based_ feature engineering using Dataflow.

In order to calculate very recent customer and terminal activity (i.e. within the last hour), computation has to be done on real-time streaming data, rather than via batch-based feature engineering. This notebook shows a step-by-step guide to create real-time data pipelines to build features. You will learn to:

- Create features, using window and aggreation functions in an Apache Beam pipeline
- Deploy the Apache Beam pipeline to Dataflow
- Ingest engineered features from Dataflow into BigQuery to serve Vertex AI Feature Store

This lab uses the following Google Cloud services and resources:

- [Pub/Sub](https://cloud.google.com/pubsub/)
- [DataFlow](https://cloud.google.com/dataflow/)
- [Vertex AI](https://cloud.google.com/vertex-ai/)
- [BigQuery](https://cloud.google.com/bigquery)

Step performed in this notebook:

- calculate customer spending features (last 15-mins, 30-mins, and 60-mins)
- calculate terminal activity features (last 15-mins, 30-mins, and 60-mins)

by pulling the streaming data from a Pub/Sub topic using the Pub/Sub subscription that we created in `00_environment_setup.ipynb` and ingesting the streaming features directly into BigQuery to serve Vertex AI Feature Store using Dataflow. 

### Load configuration settings from the setup notebook

Set the constants used in this notebook and load the config settings from the `00_environment_setup.ipynb` notebook.

In [ ]:
GCP_PROJECTS = !gcloud config get-value project
PROJECT_ID = GCP_PROJECTS[0]
BUCKET_NAME = f"{PROJECT_ID}-fraudfinder"
config = !gsutil cat gs://{BUCKET_NAME}/config/notebook_env.py
print(config.n)
exec(config.n)

### Create folder

In favour of clean folder structure, we will create a separate folder and place all the files that we will produce there.

In [ ]:
FOLDER = "./beam_pipeline_labels"
PYTHON_SCRIPT = f"{FOLDER}/main.py"
REQUIREMENTS_FILE = f"{FOLDER}/requirements.txt"

# Create new folder for pipeline files
!rm -rf {FOLDER} || True
!mkdir {FOLDER}

## Before we begin

The next Apache Beam example script programmatically sets up Google Cloud Dataflow configurations within a dictionary, passing them directly into PipelineOptions. Using these settings, it builds a simple 
Apache Beam pipeline that takes an in-memory list of four numbers and calculates their global average using a built-in combiner. 
Finally, the pipeline takes that computed average and writes it out directly to the system's standard information logs.

In [ ]:
!echo "Forcing creation of the Dataflow Service Agent..."
!gcloud beta services identity create --service=dataflow.googleapis.com --project=$PROJECT_ID
!sleep 60
!echo "Service agent is ready!"

In [ ]:
import logging
import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions

def run():
    # 1. Define your configurations programmatically
    pipeline_args = {
        'runner': 'DataflowRunner',
        'project': PROJECT_ID,
        'region': REGION,
        'temp_location': f'gs://{BUCKET_NAME}/temp',
        'job_name': 'demo-dataflow-test-job',
        'save_main_session': True # Often required when passing objects in memory
    }

    # 2. Instantiate PipelineOptions by unpacking the dictionary
    options = PipelineOptions(**pipeline_args)

    # 3. Define and run the pipeline
    with beam.Pipeline(options=options) as p:
        (
            p
            | 'CreateData' >> beam.Create([10, 20, 30, 40])
            | 'CalculateAverage' >> beam.combiners.Mean.Globally()
            | 'LogResult' >> beam.Map(lambda avg: logging.info(f'The average is: {avg}'))
        )

if __name__ == '__main__':
    logging.getLogger().setLevel(logging.INFO)
    run()

As deploying Apache Beam pipelines to Dataflow works better if we submit the job from a Python script, we will be writting the code into a python script instead of running directly on the notebook. 

In the next cells, we write the cell contents to a Python script `main.py`. We are NOT running the code direcly and an additional invocation is required. The notebook is done this way for eaiser demonstration purposes.

### Write import statements

Here we write the code to import all the required libraries to the external python script

In [ ]:
%%writefile {PYTHON_SCRIPT}

import json
import logging
import time

from typing import Tuple, Any, List

import apache_beam as beam

from apache_beam.options.pipeline_options import PipelineOptions
    
from google.cloud import aiplatform
from google.cloud import aiplatform_v1beta1

### Defining an auxiliary magic function

The magic function `writefile` from Jupyter Notebook can only write the cell as is and could not unpack Python variables. Hence, we need to create an auxiliary magic function that can unpack Python variables and write them to a file.

In [ ]:
from IPython.core.magic import register_line_cell_magic


@register_line_cell_magic
def writetemplate(line, cell):
    with open(line, "a") as f:
        f.write(cell.format(**globals()))

### Write the variable values

Here we write the variable values to the external python script using the new magic function

In [ ]:
# Adding additional variables to project_variables
project_variables = "\n".join(config[1:-1])
project_variables += f'\nPROJECT_ID = "{PROJECT}"'
project_variables += f'\nBUCKET_NAME = "{BUCKET_NAME}"'
project_variables += f'\nREQUIREMENTS_FILE = "{REQUIREMENTS_FILE}"'

In [ ]:
%%writetemplate {PYTHON_SCRIPT}

# Project variables
{project_variables}

### Write constant variables

Here we write constant variables to the external python script

In [ ]:
%%writefile -a {PYTHON_SCRIPT}

# Pub/Sub variables
LABELS_SUBSCRIPTION_NAME = "ff-txlabels-sub"
LABELS_SUBSCRIPTION_PATH = f"projects/{PROJECT_ID}/subscriptions/{LABELS_SUBSCRIPTION_NAME}"

### Building the pipeline

Now we are ready to build the pipeline using the defined classes and functions above. Once the pipeline is ready, we will wrap everything into a main function and submit it to the Dataflow.

The pipeline consists of the following steps:

1.  **Read from Pub/Sub**: The pipeline reads streaming data from a Pub/Sub subscription.
2.  **Write to BigQuery**: The prepared labels are written to a BigQuery table.

In [ ]:
%%writefile -a {PYTHON_SCRIPT}

def main():
    # Initialize Vertex AI client
    aiplatform.init(
        project=PROJECT_ID,
        location=REGION
    )

    labels_table_bq = f"{PROJECT_ID}.tx.ingestion_tx_labels"
    
    # Setup pipeline options for deploying to dataflow
    pipeline_options = PipelineOptions(streaming=True, 
                                       save_main_session=True,
                                       runner="DataflowRunner",
                                       project=PROJECT_ID,
                                       region=REGION,
                                       job_name='labels-ingestion-job',
                                       temp_location=f"gs://{BUCKET_NAME}/dataflow/tmp",
                                       requirements_file=REQUIREMENTS_FILE,
                                       max_num_workers=1)
    
    # Build pipeline and transformation steps
    pipeline = beam.Pipeline(options=pipeline_options)
    
    labels_source = (
        pipeline
        | 'Read labels from PubSub' >> beam.io.ReadFromPubSub(subscription=LABELS_SUBSCRIPTION_PATH)
        | 'Decode PubSub message' >> beam.Map(lambda row: json.loads(row.decode('utf-8')))
        | "Write labels to BQ" >> beam.io.gcp.bigquery.WriteToBigQuery(
            table=labels_table_bq,
            write_disposition=beam.io.BigQueryDisposition.WRITE_APPEND,
            create_disposition=beam.io.BigQueryDisposition.CREATE_IF_NEEDED,
        )
    )

    # Run the pipeline (async)
    pipeline.run()

    
if __name__ == "__main__":
    main()

### Creating `requirement.txt` for Dataflow Workers

As we are using `google-cloud-aiplatform` and `google-apitools` package, we need to pass the `requirement.txt` to the Dataflow Workers so that the workers will install the packages in their respective environment before running the job.

In [ ]:
%%writefile {REQUIREMENTS_FILE}

google-cloud-aiplatform<=1.36.1
google-apitools==0.5.32

### Deploying the pipeline

Now we are ready to deploy this pipeline to Dataflow.

In [ ]:
!python3 {PYTHON_SCRIPT}

Congrats! Now the job should be running on <a href="https://console.cloud.google.com/dataflow/jobs">Dataflow<a>!
    
If everything went well, you should see this Dataflow pipeline diagram on Dataflow Console.

### Verifying labels ingestion pipeline

In [ ]:
%%bigquery tx_labels_record --project {PROJECT_ID}
SELECT * 
FROM tx.ingestion_tx_labels
LIMIT 10

In [ ]:
tx_labels_record

### END

Copyright 2025 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.